In [1]:
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    DebertaForSequenceClassification,
    DebertaTokenizer,
    TrainingArguments,
    Trainer,
)

from peft import LoraConfig, get_peft_model, TaskType

import random
import numpy as np
import regex as re
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

/Users/andrejklimov/Desktop/NLP_Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0621 13:26:08.302000 78779 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0621 13:26:08.342000 78779 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0621 13:26:08.376000 78779 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
DEVICE = "mps"
MODEL = "deepvk/deberta-v1-base"

MAX_LEN = 128
BATCH_SIZE = 16
LR_FT = 2e-4
EPOCHS_FT = 2

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 1. Working with data

## 1.1 Preprocessing

In [3]:
_RE_URL = re.compile(r"https?://\S+|www\.\S+")
_RE_PUNCT = re.compile(r"[^\w\s\-]", flags=re.UNICODE)
_RE_SPACE = re.compile(r"\s+")

In [4]:
def preprocess(text):
    """
    Function to preprocess the dataset.
    Hard preprocessing would only make the data worse, so I use simple operations
    """
    text = _RE_URL.sub(" ", text)
    text = _RE_PUNCT.sub(" ", text)
    text = _RE_SPACE.sub(" ", text).strip()
    return text

## 1.2 Creating custom dataset

In [5]:
class RewiewsDataset(Dataset):
    """
    Custom dataset for processing samples
    """

    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        """
        Initialize an instance of the custom dataset
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        """
        Method to define the length
        """
        return len(self.texts)

    def __getitem__(self, idx):
        """
        Overriden method to compile each sample
        """
        tokens = self.tokenizer(
            self.texts[idx],
            return_tensors="pt",
            truncation=True,
            max_length=self.max_len,
        )

        return {
            "input_ids": tokens["input_ids"][0],
            "attention_mask": tokens["attention_mask"][0],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

# 2. Defining metrics func for evaluating model during tuning

In [6]:
def compute_metrics(eval_pred):
    """
    Method that evaluates model during DoRA fine-tuning
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1_weighted": f1}

# 3. Defining embeddings extraction func and func for training classic algos

In [7]:
@torch.no_grad()
def get_embeddings(texts, model, tokenizer):
    """
    Method that extracts embeddings to pass them through the classifier
    """
    embeddings = []
    model.eval()

    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Extracting embeddings"):
        batch = texts[i : i + BATCH_SIZE]
        tokens = tokenizer(
            batch,
            return_tensors="pt",
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
        ).to(DEVICE)
        outputs = model(**tokens)
        cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls_emb)
    return embeddings

In [8]:
def train_classifier(model_name, X_train, y_train):
    """
    Method that trains choosen classifier
    """
    if model_name == "LogisticRegression":
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(random_state=1, max_iter=1000, class_weight="balanced"),
        )
    elif model_name == "KNeighborsClassifier":
        model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=10))
    elif model_name == "SVC":
        model = make_pipeline(
            StandardScaler(),
            SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"),
        )
    elif model_name == "GaussianNB":
        model = GaussianNB()

    return model.fit(X_train, y_train)

In [9]:
def evaluate_frozen_pipeline(model, X_test, y_test):
    """
    Method that calculates various metrics for the pipeline with frozen feature extractor and classifier
    """
    y_pred = model.predict(X_test)
    return (
        f"Accuarcy: {accuracy_score(y_test, y_pred)}\n"
        f"F1 score: {f1_score(y_test, y_pred, average="weighted")}\n"
        f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}"
    )

# 4. Defining main func

In [ ]:
def main():
    """
    Main method that accumulates all the logic of the pipelines
    """
    # 1. Loading and preprocessing data
    df = pd.read_csv(
        "data/women-clothing-accessories.3-class.balanced.csv",
        engine="python",
        quoting=1,
        on_bad_lines="skip",
        sep="\t",
    )
    df["text_clean"] = df["review"].apply(preprocess)

    # 2. Encoding target
    le = LabelEncoder()
    df["label"] = le.fit_transform(df["sentiment"])
    label_names = list(le.classes_)
    print(f"Classes: {label_names}")

    # 3. Splitting data
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        df["text_clean"].tolist(),
        df["label"].tolist(),
        test_size=0.3,
        random_state=SEED,
        stratify=df["label"],
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.5, random_state=SEED, stratify=y_tmp
    )
    print(f"\nSizes: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

    print("Step 1: DeBERTa as feature extractor")
    feature_extractor = AutoModel.from_pretrained(
        pretrained_model_name_or_path=MODEL, device_map=DEVICE
    )
    feature_extractor_tokenizer = AutoTokenizer.from_pretrained(MODEL)

    train_embeds = get_embeddings(
        X_train, feature_extractor, feature_extractor_tokenizer
    )
    test_embeds = get_embeddings(X_test, feature_extractor, feature_extractor_tokenizer)

    X_train_emb = np.vstack(train_embeds)
    X_test_emb = np.vstack(test_embeds)

    print("1.1 LogisticRegression training and evaluation")
    logistic_regression = train_classifier("LogisticRegression", X_train_emb, y_train)
    print(evaluate_frozen_pipeline(logistic_regression, X_test_emb, y_test))

    print("1.2 KNN training and evaluation")
    knn_classifier = train_classifier("KNeighborsClassifier", X_train_emb, y_train)
    print(evaluate_frozen_pipeline(knn_classifier, X_test_emb, y_test))

    print("1.3 SVM training and evaluation")
    svm_classifier = train_classifier("SVC", X_train_emb, y_train)
    print(evaluate_frozen_pipeline(svm_classifier, X_test_emb, y_test))

    print("1.4 GaussianNB training and evaluation")
    gnb_classifier = train_classifier("GaussianNB", X_train_emb, y_train)
    print(evaluate_frozen_pipeline(gnb_classifier, X_test_emb, y_test))

    print("Step 2: Fine-tuning DeBERTa with classification head")
    model = DebertaForSequenceClassification.from_pretrained(
        MODEL, num_labels=3, device_map=DEVICE, ignore_mismatched_sizes=True
    )
    tokenizer = DebertaTokenizer.from_pretrained(MODEL)

    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        target_modules=["in_proj", "dense"],
        lora_dropout=0.05,
        use_dora=True,
    )

    lora_model = get_peft_model(model, lora_cfg)
    lora_model.print_trainable_parameters()

    collate_fn = DataCollatorWithPadding(tokenizer)

    train_ds = RewiewsDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = RewiewsDataset(X_val, y_val, tokenizer, MAX_LEN)
    test_ds = RewiewsDataset(X_test, y_test, tokenizer, MAX_LEN)

    training_args = TrainingArguments(
        output_dir="./dora_model",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS_FT,
        learning_rate=LR_FT,
        lr_scheduler_type="cosine",
        warmup_steps=400,
        weight_decay=0.01,
        gradient_accumulation_steps=2,
        logging_steps=500,
        eval_strategy="steps",
        fp16=False,
        dataloader_pin_memory=False,
    )

    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    test_preds = trainer.predict(test_ds)
    y_pred = np.argmax(test_preds.predictions, axis=1)
    print("\n=== Testing DoRA fine-tuned model ===")
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=1)
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 weighted: {f1:.4f}")
    print(
        classification_report(y_test, y_pred, target_names=label_names, zero_division=1)
    )
    print(confusion_matrix(y_test, y_pred))

    print(
        "Looking at the missclassified samples to evaluate the impact of the slang and other lexemas"
    )
    misclassified = []
    for i, (true, pred) in enumerate(zip(y_test, y_pred)):
        if true != pred:
            misclassified.append((X_test[i], true, pred))

    print(f"Total errors: {len(misclassified)} / {len(y_test)}")
    print("Examples of errors:")
    for text, true, pred in random.sample(misclassified, min(20, len(misclassified))):
        print(f"Text: {text[:200]}")
        print(
            f"Real label: {label_names[true]} -> Predicted label: {label_names[pred]}"
        )
        print("-" * 80)

In [11]:
main()

Classes: ['neautral', 'negative', 'positive']

Sizes: train=63000, val=13500, test=13500
Step 1: DeBERTa as feature extractor


Extracting embeddings: 100%|██████████| 844/844 [02:52<00:00,  4.90it/s]


1.1 LogisticRegression training and evaluation
Accuarcy: 0.7357777777777778
F1 score: 0.7362295300490229
Confusion Matrix:
[[2903 1029  568]
 [1165 3209  126]
 [ 584   95 3821]]
1.2 KNN training and evaluation
Accuarcy: 0.6821481481481482
F1 score: 0.6835229421746973
Confusion Matrix:
[[2760 1048  692]
 [1448 2850  202]
 [ 737  164 3599]]
1.3 SVM training and evaluation
Accuarcy: 0.748
F1 score: 0.7502358610469879
Confusion Matrix:
[[3145  904  451]
 [1281 3150   69]
 [ 607   90 3803]]
1.4 GaussianNB training and evaluation
Accuarcy: 0.6475555555555556
F1 score: 0.6504762696171884
Confusion Matrix:
[[2921  920  659]
 [1767 2510  223]
 [ 887  302 3311]]
Step 2: Fine-tuning DeBERTa with classification head


Loading weights: 100%|██████████| 160/160 [00:00<00:00, 222.23it/s]
[transformers] DebertaForSequenceClassification LOAD REPORT from: deepvk/deberta-v1-base
Key                 | Status  | 
--------------------+---------+-
classifier.bias     | MISSING | 
classifier.weight   | MISSING | 
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


trainable params: 2,469,891 || all params: 127,106,310 || trainable%: 1.9432


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,Accuracy,F1 Weighted
500,1.487935,0.584441,0.750444,0.750112
1000,1.159910,0.559094,0.755704,0.759119
1500,1.112190,0.536439,0.763852,0.763413
2000,1.085735,0.542782,0.763704,0.764654
2500,1.027336,0.516475,0.771407,0.773422
3000,1.017339,0.517478,0.772222,0.773740
3500,1.000025,0.513953,0.775407,0.777582
3938,1.000025,0.514350,0.776519,0.778492



=== Testing DoRA fine-tuned model ===
Accuracy: 0.7791
F1 weighted: 0.7807
              precision    recall  f1-score   support

    neautral       0.67      0.70      0.69      4500
    negative       0.76      0.76      0.76      4500
    positive       0.92      0.87      0.89      4500

    accuracy                           0.78     13500
   macro avg       0.78      0.78      0.78     13500
weighted avg       0.78      0.78      0.78     13500

[[3155 1017  328]
 [1032 3435   33]
 [ 517   55 3928]]
Looking at the missclassified samples to evaluate the impact of the slang and other lexemas
Total errors: 2982 / 13500
Examples of errors:
Text: рубашка хорошая но на мой 48-50 3хL мала
Real label: neautral -> Predicted label: positive
--------------------------------------------------------------------------------
Text: Продавец ошибся в размере Курточка хорошая но очень маленькая Я заказывала размер42 размер но получила размер
Real label: negative -> Predicted label: neautral
-----